# 5.4 · El fallo provocado: depurar con lenguaje

**Objetivo:** en un agente, muchos bugs **no están en el código** sino en el lenguaje (descripciones de herramientas, system prompt), y se diagnostican **leyendo trazas**.

Este notebook parte de una copia del 5.3 con un fallo plantado. No mires todavía las descripciones de las herramientas: primero, el síntoma.

## Preparación

In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv()

import sys; sys.path.append("../..")   # para importar desde la raíz del taller
from pathlib import Path
from config import MODEL

import json
import herramientas
from claude_agent_sdk import (
    query, tool, create_sdk_mcp_server, ClaudeAgentOptions, AgentDefinition, HookMatcher,
    ResultMessage, ToolAnnotations,
)
from taller_utils import imprimir, buscar_kb, auditar, CONTEXTO_LOG, leer_log, borrar_log, restaurar_datos

CORREOS = herramientas.listar_bandeja()[:5]   # el 006 es para el bloque 6
CORREOS

In [ ]:
def texto(dato):
    if not isinstance(dato, str):
        dato = json.dumps(dato, ensure_ascii=False)
    return {"content": [{"type": "text", "text": dato}]}

def error(mensaje):
    return {"content": [{"type": "text", "text": mensaje}], "is_error": True}

@tool("leer_correo",
      "Lee un correo de la bandeja de soporte a partir de su nombre de fichero "
      "(p. ej. '001_exportar_excel.txt'). Devuelve el texto completo: remitente, asunto y cuerpo.",
      {"nombre": str},
      annotations=ToolAnnotations(readOnlyHint=True))
async def leer_correo(args):
    try:
        return texto(herramientas.leer_correo(args["nombre"]))
    except FileNotFoundError as e:
        return error(str(e))

In [ ]:
# ❌ FALLO PROVOCADO: las descripciones de esta versión
@tool("buscar_incidencias",
      "Busca incidencias.",
      {
          "type": "object",
          "properties": {
              "cliente": {"type": "string", "description": "Nombre del cliente tal como firma el correo, "
                                                           "p. ej. 'ACME Corporation'"},
              "estado": {"type": "string", "enum": ["abierta", "cerrada"]},
          },
          "required": ["cliente"],
      },
      annotations=ToolAnnotations(readOnlyHint=True))
async def buscar_incidencias(args):
    return texto(herramientas.buscar_incidencias(**args))

@tool("crear_incidencia",
      "Crea una incidencia nueva y devuelve la incidencia creada con su id.",
      {
          "type": "object",
          "properties": {
              "cliente": {"type": "string", "description": "Nombre del cliente tal como firma el correo"},
              "asunto": {"type": "string", "description": "Resumen en menos de 10 palabras"},
              "descripcion": {"type": "string"},
              "prioridad": {"type": "string", "enum": ["alta", "media", "baja"]},
          },
          "required": ["cliente", "asunto", "descripcion", "prioridad"],
      })
async def crear_incidencia(args):
    return texto(herramientas.crear_incidencia(**args))

from pydantic import BaseModel, Field
from typing import Literal

class Triaje(BaseModel):
    correo: str
    es_incidencia: bool = Field(description="True si el remitente es un cliente; False si no lo es")
    cliente: str | None
    incidencia_id: str | None
    accion: Literal["creada", "existente", "ninguna"]
    prioridad: Literal["alta", "media", "baja"] | None
    articulo_kb: str | None
    respuesta_propuesta: str

In [ ]:
CRITERIOS = Path("../../datos/kb/prioridades.md").read_text(encoding="utf-8")

clasificador = AgentDefinition(
    description=("Especialista en clasificar incidencias de soporte. Úsalo cuando haya que "
                 "decidir la prioridad de una incidencia nueva."),
    prompt=("Eres un clasificador de incidencias. Dada la descripción de un problema, aplica estos "
            "criterios y devuelve SOLO: prioridad (alta/media/baja), categoría "
            "(facturación/técnico/cuenta/otro) y una justificación de una línea.\n\n" + CRITERIOS),
    tools=[],   # no necesita herramientas: los criterios van en su prompt
    model="claude-haiku-5-5",
)

In [ ]:
SYSTEM = """Eres un asistente de triaje de soporte. Para el correo indicado:
1. Lee el correo e identifica al cliente.
2. Registra la incidencia correspondiente, con la prioridad que indique el subagente clasificador.
3. Busca en la base de conocimiento un artículo que ayude y propón una respuesta basada SOLO en él.
Nunca envías nada: tu salida la revisa una persona."""

In [ ]:
soporte = create_sdk_mcp_server(name="soporte", version="1.0.0",
                                tools=[leer_correo, buscar_incidencias, crear_incidencia, buscar_kb])

options = ClaudeAgentOptions(
    model=MODEL,
    system_prompt=SYSTEM,
    mcp_servers={"soporte": soporte},
    tools=["Task"],
    allowed_tools=["Task", "mcp__soporte__leer_correo", "mcp__soporte__buscar_incidencias",
                   "mcp__soporte__crear_incidencia", "mcp__soporte__buscar_kb"],
    agents={"clasificador": clasificador},
    hooks={
        "PreToolUse":  [HookMatcher(hooks=[auditar])],
        "PostToolUse": [HookMatcher(hooks=[auditar])],
    },
    output_format={"type": "json_schema", "schema": Triaje.model_json_schema()},
    env={"CLAUDE_CODE_DISABLE_BACKGROUND_TASKS": "1"},   # que espere al subagente (4.3)
    max_turns=15,
    setting_sources=[],
)

In [ ]:
async def triar(nombre_correo, **etiquetas):
    CONTEXTO_LOG.clear()
    CONTEXTO_LOG.update(correo=nombre_correo, **etiquetas)   # cada línea del log lleva el correo
    async for msg in query(prompt=f"Tría el correo {nombre_correo}", options=options):
        imprimir(msg)
        if isinstance(msg, ResultMessage):
            return msg

In [ ]:
def trazas(correo, **filtros):
    log = leer_log()
    log = log[(log.correo == correo) & (log.evento == "PreToolUse") & (log.herramienta != "StructuredOutput")]
    for campo, valor in filtros.items():
        log = log[log[campo] == valor]
    return log[["herramienta", "entrada"]]

## El síntoma

El correo duplicado: Initech vuelve a escribir sobre la sincronización con Outlook, que ya tiene la incidencia `INC-008` abierta.

In [ ]:
print(herramientas.leer_correo("002_duplicado.txt"))

In [ ]:
restaurar_datos()
borrar_log()

r = await triar("002_duplicado.txt", version="antes")
r.structured_output

In [ ]:
herramientas.buscar_incidencias(texto="Outlook")

`accion: "creada"`, y hay **dos** incidencias sobre lo mismo. Tenemos un bug. ¿Dónde está?

> Si en tu ejecución no ha fallado, vuelve a ejecutar las dos celdas anteriores: los agentes no son deterministas. Ese es otro motivo para no fiarse de una sola prueba.

## La tentación: arreglarlo con código

El parche típico, en `crear_incidencia`:

```python
import difflib

def crear_incidencia(cliente, asunto, descripcion, prioridad):
    for inc in buscar_incidencias(estado="abierta"):
        if difflib.SequenceMatcher(None, inc["asunto"], asunto).ratio() > 0.6:
            return inc          # "ya existe"
    ...
```

Tapa el síntoma, pero es frágil (¿y si el asunto está escrito con otras palabras?, ¿y si son dos clientes distintos?), no generaliza y no explica **por qué** el agente decidió crear.

## Diagnóstico con trazas

Las llamadas a herramientas de este correo, del log de auditoría:

In [ ]:
trazas("002_duplicado.txt", version="antes")

Lo que suele mostrar:

- Buscó con `cliente: "Initech S.L."`, el resultado fue `[]` y creó una nueva. O bien
- no hay ninguna llamada a `buscar_incidencias`: **nunca buscó**.

La pregunta clave: **con la información que tenía el modelo, ¿era razonable lo que hizo?** Leamos lo que tenía:

In [ ]:
print(buscar_incidencias.description)
print(crear_incidencia.description)
print(SYSTEM)

Y el esquema de la búsqueda:

In [ ]:
buscar_incidencias.input_schema

Sí, era razonable. **Nadie le dijo que tenía que buscar antes de crear, ni cómo buscar.** El paso 2 del prompt le dice que registre la incidencia, sin más; la descripción de `buscar_incidencias` no dice cuándo usarla, el esquema solo permite buscar por cliente (no hay filtro de texto), lo hace obligatorio y pide el nombre *tal como firma el correo*, con un ejemplo que incluye la forma jurídica: "Initech S.L.". En los datos, el cliente es "Initech". El modelo hizo exactamente lo que le dijimos.

## Arreglo: iterar sobre el lenguaje

Tres cambios, todos de texto (el esquema también es texto que lee el modelo):

1. **Descripción y esquema de `buscar_incidencias`:** cuándo usarla (*antes de crear cualquier incidencia*), qué devuelve y qué hacer si no encuentra nada. En el esquema, el `cliente` deja de ser obligatorio y se describe como el nombre comercial corto, y vuelve el filtro `texto`.
2. **Descripción de `crear_incidencia`:** *úsala SOLO después de haber comprobado con buscar_incidencias…*
3. **System prompt:** recuperar el paso de buscar.

In [ ]:
@tool("buscar_incidencias",
      "Busca incidencias de soporte existentes. Úsala SIEMPRE antes de crear cualquier incidencia, para "
      "comprobar si ya hay una abierta sobre el mismo problema. Todos los filtros son opcionales y se "
      "combinan. Devuelve una lista JSON con id, cliente, asunto, estado, prioridad y fecha; [] si no "
      "hay coincidencias. Si buscas por cliente y no hay resultados, prueba también con `texto` usando "
      "palabras clave del problema.",
      {
          "type": "object",
          "properties": {
              "texto": {"type": "string", "description": "Palabras clave a buscar en asunto y descripción"},
              "estado": {"type": "string", "enum": ["abierta", "cerrada"]},
              "cliente": {"type": "string", "description": "Nombre comercial corto del cliente, sin forma "
                                                           "jurídica: 'Initech', no 'Initech S.L.'"},
          },
      },
      annotations=ToolAnnotations(readOnlyHint=True))
async def buscar_incidencias(args):
    return texto(herramientas.buscar_incidencias(**args))

@tool("crear_incidencia",
      "Crea una incidencia nueva y devuelve la incidencia creada con su id. Úsala SOLO después "
      "de haber comprobado con buscar_incidencias que no existe ya una incidencia abierta sobre el mismo "
      "problema para el mismo cliente. Si existe, no crees nada: usa la existente.",
      {
          "type": "object",
          "properties": {
              "cliente": {"type": "string", "description": "Nombre comercial corto del cliente, sin forma jurídica"},
              "asunto": {"type": "string", "description": "Resumen en menos de 10 palabras"},
              "descripcion": {"type": "string", "description": "Qué ocurre, desde cuándo y a quién afecta"},
              "prioridad": {"type": "string", "enum": ["alta", "media", "baja"]},
          },
          "required": ["cliente", "asunto", "descripcion", "prioridad"],
      })
async def crear_incidencia(args):
    return texto(herramientas.crear_incidencia(**args))

In [ ]:
SYSTEM = """Eres un asistente de triaje de soporte. Para el correo indicado:
1. Lee el correo e identifica al cliente.
2. Busca si ya existe una incidencia abierta sobre el mismo problema para ese cliente.
3. Si no existe, registra la incidencia correspondiente, con la prioridad que indique el subagente clasificador.
4. Busca en la base de conocimiento un artículo que ayude y propón una respuesta basada SOLO en él.
Nunca envías nada: tu salida la revisa una persona."""

Reejecutamos **solo** lo necesario: las opciones (que recogen las herramientas y el prompt nuevos), el correo duplicado y las trazas.

In [ ]:
soporte = create_sdk_mcp_server(name="soporte", version="1.0.0",
                                tools=[leer_correo, buscar_incidencias, crear_incidencia, buscar_kb])

options = ClaudeAgentOptions(
    model=MODEL,
    system_prompt=SYSTEM,
    mcp_servers={"soporte": soporte},
    tools=["Task"],
    allowed_tools=["Task", "mcp__soporte__leer_correo", "mcp__soporte__buscar_incidencias",
                   "mcp__soporte__crear_incidencia", "mcp__soporte__buscar_kb"],
    agents={"clasificador": clasificador},
    hooks={
        "PreToolUse":  [HookMatcher(hooks=[auditar])],
        "PostToolUse": [HookMatcher(hooks=[auditar])],
    },
    output_format={"type": "json_schema", "schema": Triaje.model_json_schema()},
    env={"CLAUDE_CODE_DISABLE_BACKGROUND_TASKS": "1"},   # que espere al subagente (4.3)
    max_turns=15,
    setting_sources=[],
)

In [ ]:
restaurar_datos()

r = await triar("002_duplicado.txt", version="después")
r.structured_output

Antes y después:

In [ ]:
trazas("002_duplicado.txt", version="antes")

In [ ]:
trazas("002_duplicado.txt", version="después")

Ahora aparece la búsqueda (o dos, si la primera no encuentra nada) antes de decidir, y la acción es `existente`.

## El método

1. **Reproducir** el fallo.
2. **Leer las trazas:** ¿qué decidió y con qué información?
3. **Preguntar** si la decisión era razonable con esa información.
4. Si lo era, arreglar **el lenguaje**. Si no, arreglar el código o cambiar de modelo.
5. **Reejecutar y comparar** las trazas.

Cambia una sola cosa cada vez: si cambias tres y funciona, no sabes cuál lo arregló (ni si alguna empeoró otra cosa).

In [ ]:
restaurar_datos()

## 🔸 Reto

Hay un segundo fallo plantado: el correo de felicitación (`005_felicitacion.txt`) sale con `es_incidencia: true`, así que no cumple el tercer criterio del 5.1. Encuéntralo y arréglalo **sin tocar la lógica en Python**: solo los textos que lee el modelo (descripciones, esquemas y prompt).